# Query Generation
## Stage 2 of the Dataset Discovery Pipeline

This notebook runs **Query Generation** on `output/seed_signature.json` and produces
`output/queries.json` — a ranked list of search queries used by Stage 3 (Crawler)
to discover joinable datasets on the web.

---
### What this notebook does
1. Loads `output/seed_signature.json` produced by Stage 1
2. Displays the key signals extracted from the seed dataset
3. Runs the query generator (5 strategies, **normal mode**)
4. Shows all generated queries grouped by strategy
5. Summarises query counts by strategy and concept
6. Writes `output/queries.json`
7. Checks discovery sufficiency thresholds
8. uns fallback mode if results are insufficient (tiered relaxation)**

> **Collaborators:** run all cells top-to-bottom (`Runtime → Run all`).  
> Stage 1 must have been run first — `output/seed_signature.json` must exist.


## 1. Environment Setup

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "-r", "../requirements.txt", "-q"])
print("Dependencies ready.")

Dependencies ready.


In [2]:
import sys, os, json
from pathlib import Path
import pandas as pd

# Make src/ importable whether running from notebooks/ or root
for candidate in ["../src", "src"]:
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)

from query_generator import generate, save_queries, generate_fallback, check_sufficiency, SUFFICIENCY_THRESHOLDS
print("Imports OK.")


Imports OK.


## 2. Load the Seed Signature

This is the output of Stage 1 (`profiler.py`). It contains everything the
query generator needs: inferred concepts, candidate identifiers, entity value
samples, salient column names, and year range.

In [3]:
# Resolve path regardless of whether notebook is opened from notebooks/ or root
_here = Path(".").resolve()
if (_here / "output" / "seed_signature.json").exists():
    SIG_PATH    = _here / "output" / "seed_signature.json"
    OUTPUT_PATH = _here / "output" / "queries.json"
else:
    SIG_PATH    = _here.parent / "output" / "seed_signature.json"
    OUTPUT_PATH = _here.parent / "output" / "queries.json"

with open(SIG_PATH, "r", encoding="utf-8") as f:
    sig = json.load(f)

print(f"Loaded  : {SIG_PATH}")
print(f"Dataset : {sig['dataset_name']}")
print(f"Rows    : {sig['n_rows']}")
print(f"Columns : {sig['n_columns']}")

Loaded  : C:\Users\kalle\Desktop\Seed-and-Seek\output\seed_signature.json
Dataset : seed.csv
Rows    : 32
Columns : 14


## 3. Key Signals from the Seed Signature

These are the inputs the query generator uses to build queries.

In [4]:
print("── Inferred concepts ──────────────────────────────────────")
print("  " + ", ".join(sig["inferred_concepts"]))

print("\n── Candidate identifiers (join keys) ──────────────────────")
print("  " + ", ".join(sig["candidate_identifiers"]))

print("\n── Salient column names ───────────────────────────────────")
print("  " + ", ".join(sig["salient_column_names"]))

print("\n── Entity value samples ───────────────────────────────────")
for col, vals in sig["entity_value_samples"].items():
    print(f"  {col}: {vals}")

── Inferred concepts ──────────────────────────────────────
  demographics, economic, education, environmental, geospatial, public_health, time_series

── Candidate identifiers (join keys) ──────────────────────
  country_code, currency_code

── Salient column names ───────────────────────────────────
  country_code, country_name, year, population, region, income_group, currency_code

── Entity value samples ───────────────────────────────────
  country_name: ['United States', 'Germany', 'Brazil', 'India', 'Nigeria']
  region: ['North America', 'Europe', 'Latin America', 'South Asia', 'Sub-Saharan Africa']


## 4. Run the Query Generator

In [5]:
output = generate(SIG_PATH)
save_queries(output, OUTPUT_PATH)

print(f"\nMode                    : {output['mode']}")
print(f"Total queries generated : {output['total_queries']}")
print(f"Year range              : {output['year_range']}")
print(f"Target connectors       : {output['target_connectors']}")
print(f"Output                  : {OUTPUT_PATH}")


[query_generator] Concepts     : ['demographics', 'economic', 'education', 'environmental', 'geospatial', 'public_health', 'time_series']
[query_generator] Identifiers  : ['country_code', 'currency_code']
[query_generator] Year range   : 2020 – 2022
[query_generator] Columns      : ['country_code', 'country_name', 'year', 'gdp_usd', 'gdp_growth_pct', 'population', 'life_expectancy', 'infant_mortality_rate', 'literacy_rate_pct', 'urban_pop_pct', 'co2_emissions_kt', 'region', 'income_group', 'currency_code']
[query_generator] Total queries generated: 346
[query_generator] Queries written to C:\Users\kalle\Desktop\Seed-and-Seek\output\queries.json

Total queries generated : 346
Year range              : [2020, 2022]
Target connectors       : ['CKAN', 'Schema.org', 'DCAT']
Output                  : C:\Users\kalle\Desktop\Seed-and-Seek\output\queries.json


## 5. Queries by Strategy

The generator uses 5 strategies in **normal mode**:

| Strategy | Description | Priority |
|---|---|---|
| `direct_platform` | Targets CKAN portals, Schema.org datasets, DCAT catalogs | 1 |
| `concept_x_source` | Combines inferred concepts with known data sources | 1 |
| `joinability` | Anchored on candidate identifiers (e.g. `country_code`) | 2 |
| `synonym` | Alternative terminology for existing seed columns | 2 |
| `augmentation` | Targets columns *not* already in the seed dataset | 3 |

In **fallback mode**, two additional relaxed strategies are used:

| Strategy | Description | Tier | Priority |
|---|---|---|---|
| `fallback_tier1_no_year` | Same concepts/sources but year range dropped | 1 | 4 |
| `fallback_tier1_join_no_year` | Joinability queries without year constraint | 1 | 4 |
| `fallback_tier2_extended_sources` | Bare topic queries on Kaggle, HuggingFace, World Bank, etc. | 2 | 5 |
| `fallback_tier2_synonym_extended` | Synonym queries on extended sources | 2 | 5 |
| `fallback_tier2_join_extended` | Joinability queries on extended sources | 2 | 5 |


In [6]:
queries_df = pd.DataFrame(output["queries"])

display_cols = [c for c in ["id", "priority", "strategy", "target_connector", "concept", "query"] if c in queries_df.columns]
queries_df[display_cols]

,id,priority,strategy,target_connector,concept,query
0,1,1,direct_platform,CKAN,demographics,CKAN demographics country year dataset
1,2,1,direct_platform,Schema.org,demographics,Schema.org dataset demographics by country 202...
2,3,1,direct_platform,DCAT,demographics,DCAT catalog demographics country panel data
3,4,1,direct_platform,CKAN,economic,CKAN economic country year dataset
4,5,1,direct_platform,Schema.org,economic,Schema.org dataset economic by country 2020 2022
...,...,...,...,...,...,...
341,342,3,augmentation,Schema.org,time_series,yearly indicators Europe Schema.org dataset 20...
342,343,3,augmentation,CKAN,time_series,longitudinal data North America CKAN 2020 2022
343,344,3,augmentation,Schema.org,time_series,longitudinal data North America Schema.org dat...
344,345,3,augmentation,CKAN,time_series,longitudinal data Europe CKAN 2020 2022


In [7]:
# Show each strategy separately
all_strategies = [
    "direct_platform", "concept_x_source", "joinability", "synonym", "augmentation",
    "fallback_tier1_no_year", "fallback_tier1_join_no_year",
    "fallback_tier2_extended_sources", "fallback_tier2_synonym_extended", "fallback_tier2_join_extended",
]
for strategy in all_strategies:
    subset = queries_df[queries_df["strategy"] == strategy]
    if subset.empty:
        continue
    print(f"\n── {strategy} ({len(subset)} queries) ──────────────────────")
    for _, row in subset.head(5).iterrows():
        print(f"  [{row['id']:>3}] [{row['target_connector']}] {row['query']}")
    if len(subset) > 5:
        print(f"  ... and {len(subset) - 5} more")



── direct_platform (21 queries) ──────────────────────
  [  1] [CKAN] CKAN demographics country year dataset
  [  2] [Schema.org] Schema.org dataset demographics by country 2020 2022
  [  3] [DCAT] DCAT catalog demographics country panel data
  [  4] [CKAN] CKAN economic country year dataset
  [  5] [Schema.org] Schema.org dataset economic by country 2020 2022
  ... and 16 more

── concept_x_source (81 queries) ──────────────────────
  [ 22] [CKAN] population growth rate CKAN 2020 2022
  [ 23] [Schema.org] population growth rate Schema.org dataset 2020 2022
  [ 24] [DCAT] population growth rate DCAT catalog 2020 2022
  [ 25] [CKAN] age distribution CKAN 2020 2022
  [ 26] [Schema.org] age distribution Schema.org dataset 2020 2022
  ... and 76 more

── joinability (24 queries) ──────────────────────
  [103] [CKAN] country code by country by year CKAN 2020 2022
  [104] [Schema.org] country code by country by year Schema.org dataset 2020 2022
  [105] [DCAT] country code by country by year

## 6. Summary

In [8]:
print("── Queries per strategy ───────────────────────────────────")
strategy_counts = queries_df["strategy"].value_counts()
for strategy, count in strategy_counts.items():
    print(f"  {strategy:30s} {count}")
print(f"\n  {'TOTAL':30s} {len(queries_df)}")

── Queries per strategy ───────────────────────────────────
  augmentation                   128
  synonym                        90
  concept_x_source               81
  joinability                    24
  direct_platform                21
  joinability_entity_sample      2

  TOTAL                          346


In [9]:
print("── Queries per target connector ───────────────────────────")
connector_counts = queries_df["target_connector"].value_counts()
for connector, count in connector_counts.items():
    print(f"  {connector:30s} {count}")

── Queries per target connector ───────────────────────────
  CKAN                           137
  Schema.org                     137
  DCAT                           72


In [10]:
print("── Queries per concept ────────────────────────────────────")
if "concept" in queries_df.columns:
    concept_counts = queries_df["concept"].dropna().value_counts()
    for concept, count in concept_counts.items():
        print(f"  {concept:30s} {count}")
else:
    print("  No concept field found.")

── Queries per concept ────────────────────────────────────
  economic                       43
  public_health                  39
  demographics                   35
  environmental                  35
  education                      31
  time_series                    24
  geospatial                     23


## 7. Output

The queries have been written to `output/queries.json`.

```
output/
├── seed_signature.json        ← Stage 1 output (profiler)
├── queries.json               ← Stage 2 output (normal mode)
└── queries_fallback.json      ← Stage 2 output (fallback mode, if triggered)
```

**Next step:** Stage 3 (Crawler) will read `queries.json` and use the
`target_connector` field to route each query to the correct ecosystem:
CKAN portals, Schema.org pages, or DCAT catalogs.

After Stage 3 runs, it writes `output/discovery_feedback.json`. If results
are insufficient, run **Section 8** below to trigger fallback query generation.


## 8. Sufficiency Check

After Stage 3 (Crawler) runs, it produces `output/discovery_feedback.json`
with the following fields:

```json
{
    "total_candidates":  int,    // how many datasets were found
    "joinable_count":    int,    // how many have a usable join key
    "avg_relevance":     float,  // average relevance score (0–1)
    "connectors_tried":  [...],  // which connectors were queried
    "failed_connectors": [...]   // connectors that returned 0 results
}
```

This cell checks whether those results meet the minimum thresholds.
If they do not, **Section 9** generates relaxed fallback queries.


In [ ]:
print("── Sufficiency thresholds ─────────────────────────────────")
for key, value in SUFFICIENCY_THRESHOLDS.items():
    print(f"  {key:25s} {value}")


In [ ]:
FEEDBACK_PATH = OUTPUT_PATH.parent / "discovery_feedback.json"

if not FEEDBACK_PATH.exists():
    print(f"[!] No feedback file found at {FEEDBACK_PATH}")
    print("    Run Stage 3 (Crawler) first, then re-run this cell.")
    sufficiency = None
else:
    with open(FEEDBACK_PATH, "r", encoding="utf-8") as f:
        feedback = json.load(f)

    sufficiency = check_sufficiency(feedback)

    print("── Discovery feedback ─────────────────────────────────────")
    print(f"  total_candidates  : {feedback.get('total_candidates', 'N/A')}")
    print(f"  joinable_count    : {feedback.get('joinable_count', 'N/A')}")
    print(f"  avg_relevance     : {feedback.get('avg_relevance', 'N/A')}")
    print()
    if sufficiency["is_sufficient"]:
        print("✅ Results are sufficient — fallback not needed.")
    else:
        print("⚠️  Results are insufficient. Fallback will be triggered.")
        print(f"   Relaxation level : {sufficiency['relaxation_level']} / 2")
        print(f"   Failed checks    :")
        for check in sufficiency["failed_checks"]:
            print(f"     • {check}")


## 9. Fallback Query Generation

Only runs if the sufficiency check above detected insufficient results.

- **Relaxation level 1** — drops the year range, expands to all topics per concept (same 3 connectors)
- **Relaxation level 2** — additionally switches to extended sources: Kaggle, Hugging Face, World Bank, UN Data, Eurostat, Our World in Data

The output is written to `output/queries_fallback.json` with the same structure
as `queries.json`, so Stage 3 can consume it without any changes.


In [ ]:
FALLBACK_OUTPUT_PATH = OUTPUT_PATH.parent / "queries_fallback.json"

if sufficiency is None:
    print("[!] Run Section 8 first to load the feedback file.")
elif sufficiency["is_sufficient"]:
    print("✅ Discovery results are sufficient — fallback not needed.")
else:
    print(f"Running fallback at relaxation level {sufficiency['relaxation_level']}...")
    fallback_output = generate_fallback(SIG_PATH, FEEDBACK_PATH)

    if fallback_output:
        save_queries(fallback_output, FALLBACK_OUTPUT_PATH)
        print(f"\nFallback mode           : {fallback_output['mode']}")
        print(f"Relaxation level        : {fallback_output['relaxation_level']}")
        print(f"Failed checks           : {fallback_output['failed_checks']}")
        print(f"Total fallback queries  : {fallback_output['total_queries']}")
        print(f"Target connectors       : {fallback_output['target_connectors']}")
        print(f"Output                  : {FALLBACK_OUTPUT_PATH}")


In [ ]:
if sufficiency and not sufficiency["is_sufficient"]:
    fallback_df = pd.DataFrame(fallback_output["queries"])
    display_cols = [c for c in ["id", "priority", "fallback_tier", "strategy", "target_connector", "query"] if c in fallback_df.columns]

    print("── Fallback queries per strategy ──────────────────────────")
    for strategy, count in fallback_df["strategy"].value_counts().items():
        print(f"  {strategy:45s} {count}")
    print(f"\n  {'TOTAL':45s} {len(fallback_df)}")

    print("\n── Fallback queries per connector ─────────────────────────")
    for connector, count in fallback_df["target_connector"].value_counts().items():
        print(f"  {connector:45s} {count}")

    print("\n── Sample fallback queries ────────────────────────────────")
    for tier in [1, 2]:
        tier_df = fallback_df[fallback_df.get("fallback_tier", pd.Series()) == tier] if "fallback_tier" in fallback_df.columns else pd.DataFrame()
        if not tier_df.empty:
            print(f"\n  Tier {tier}:")
            for _, row in tier_df.head(4).iterrows():
                print(f"    [{row['id']:>3}] [{row['target_connector']}] {row['query']}")
